In [1]:
import pandas as pd
import re


In [2]:
postings = pd.read_csv("Data Sets/raw/postings.csv")
postings.head()


,job_title,company,job_location,job_link,first_seen,search_city,search_country,job level,job_type,job_summary,job_skills
0,"Data Analyst-SQL, Tableau",Zortech Solutions,"Mountain View, CA",https://www.linkedin.com/jobs/data-analyst-jobs,2023-12-20,Bloomington,United States,Associate,Onsite,NaN,NaN
1,Market Research & Insights Analyst,Indiana University Foundation,"Bloomington, IN",https://www.linkedin.com/jobs/view/market-rese...,2023-12-20,Bloomington,United States,Mid senior,Onsite,Company Description\nAre you a high-performer ...,"Data analysis, Market research, Survey develop..."
2,Business Systems Analyst `1,Cook Medical,"Bloomington, IN",https://www.linkedin.com/jobs/view/business-sy...,2023-12-20,Bloomington,United States,Mid senior,Onsite,Overview\nThe Business Systems Analyst 1 perfo...,"Business Analysis, Technical Writing, Software..."
3,Senior VAT and Indirect Tax Analyst,Epic,"Bloomington, IN",https://www.linkedin.com/jobs/view/senior-vat-...,2023-12-20,Bloomington,United States,Mid senior,Onsite,We're looking for an experienced tax professio...,"Accounting, Finance, VAT/GST tax regimes, US a..."
4,Senior HRIS Analyst (Timekeeping and Payroll),Nordson Corporation,Greater Bloomington Area,https://www.linkedin.com/jobs/view/senior-hris...,2023-12-20,Bloomington,United States,Mid senior,Remote,Collaboration drives Nordson’s success as a ma...,"Workday HCM, UKG Dimensions, Ceridian Dayforce..."


In [3]:
print(postings.shape)
print(postings.columns)
postings.info()


(12894, 11)
Index(['job_title', 'company', 'job_location', 'job_link', 'first_seen',
       'search_city', 'search_country', 'job level', 'job_type', 'job_summary',
       'job_skills'],
      dtype='object')
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12894 entries, 0 to 12893
Data columns (total 11 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   job_title       12894 non-null  object
 1   company         12894 non-null  object
 2   job_location    12894 non-null  object
 3   job_link        12894 non-null  object
 4   first_seen      12894 non-null  object
 5   search_city     12894 non-null  object
 6   search_country  12894 non-null  object
 7   job level       12894 non-null  object
 8   job_type        12894 non-null  object
 9   job_summary     12851 non-null  object
 10  job_skills      12705 non-null  object
dtypes: object(11)
memory usage: 1.1+ MB


In [4]:
def clean_text(x):
    x = "" if pd.isna(x) else str(x)
    x = x.strip()
    x = re.sub(r"\s+", " ", x)
    return x

postings_clean = postings.copy()

for col in ["job_title", "company", "job_location", "job_level", "job_type", "job_summary", "job_skills"]:
    if col in postings_clean.columns:
        postings_clean[col] = postings_clean[col].apply(clean_text)


In [5]:
if "job_summary" in postings_clean.columns:
    postings_clean = postings_clean.rename(columns={"job_summary": "job_description"})


In [6]:
postings_clean["job_title"] = postings_clean["job_title"].replace("", pd.NA)
postings_clean["job_skills"] = postings_clean["job_skills"].replace("", pd.NA)

postings_clean = postings_clean.dropna(subset=["job_title", "job_skills"])
print("Rows after removing missing essentials:", len(postings_clean))


Rows after removing missing essentials: 12705


In [7]:
if "job_link" in postings_clean.columns:
    postings_clean = postings_clean.drop_duplicates(subset=["job_link"])
else:
    postings_clean = postings_clean.drop_duplicates()


In [8]:
postings_clean["job_skills_list"] = postings_clean["job_skills"].apply(
    lambda x: [s.strip().lower() for s in str(x).split(",") if s.strip()]
)
postings_clean[["job_title", "job_skills", "job_skills_list"]].head(5)


,job_title,job_skills,job_skills_list
1,Market Research & Insights Analyst,"Data analysis, Market research, Survey develop...","[data analysis, market research, survey develo..."
2,Business Systems Analyst `1,"Business Analysis, Technical Writing, Software...","[business analysis, technical writing, softwar..."
3,Senior VAT and Indirect Tax Analyst,"Accounting, Finance, VAT/GST tax regimes, US a...","[accounting, finance, vat/gst tax regimes, us ..."
4,Senior HRIS Analyst (Timekeeping and Payroll),"Workday HCM, UKG Dimensions, Ceridian Dayforce...","[workday hcm, ukg dimensions, ceridian dayforc..."
5,Business Intelligence Reporting Analyst 2,"SAP Business Objects, SQL, Qlik, Data Modeling...","[sap business objects, sql, qlik, data modelin..."


In [9]:
postings_clean["job_text"] = (
    postings_clean["job_title"].fillna("") + " " +
    postings_clean["job_description"].fillna("") + " " +
    postings_clean["job_skills"].fillna("")
)


In [10]:
postings_clean.to_csv("Data Sets/cleaned/postings_clean.csv", index=False)
print("Saved final dataset ✅")


Saved final dataset ✅


In [11]:
def match_resume_to_job(resume_skills, job_skills):
    resume_set = set([s.lower().strip() for s in resume_skills])
    job_set = set(job_skills)

    matched = resume_set.intersection(job_set)

    score = len(matched) / len(job_set) if len(job_set) > 0 else 0
    return score, matched


In [12]:
resume_skills = ["python", "sql", "excel", "tableau"]

postings_clean["match_score"] = postings_clean["job_skills_list"].apply(
    lambda x: match_resume_to_job(resume_skills, x)[0]
)

postings_clean[["job_title", "match_score"]].sort_values(
    by="match_score", ascending=False
).head(10)


,job_title,match_score
1640,Data Analyst,1.000000
1039,Senior Data Analyst,0.666667
5294,Data Analyst-III #: 23-06518,0.600000
736,Senior Financial Analyst,0.500000
8322,Healthcare Data Analyst,0.500000
2846,Financial Systems Analyst,0.500000
2277,Sr Data/Business Analyst,0.500000
9645,Treasury Analyst Senior,0.500000
72,Senior Data Analyst,0.500000
6799,Data Analyst I,0.500000


In [13]:
postings_clean.info()
postings_clean.isna().sum()


<class 'pandas.core.frame.DataFrame'>
Index: 12705 entries, 1 to 12893
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   job_title        12705 non-null  object 
 1   company          12705 non-null  object 
 2   job_location     12705 non-null  object 
 3   job_link         12705 non-null  object 
 4   first_seen       12705 non-null  object 
 5   search_city      12705 non-null  object 
 6   search_country   12705 non-null  object 
 7   job level        12705 non-null  object 
 8   job_type         12705 non-null  object 
 9   job_description  12705 non-null  object 
 10  job_skills       12705 non-null  object 
 11  job_skills_list  12705 non-null  object 
 12  job_text         12705 non-null  object 
 13  match_score      12705 non-null  float64
dtypes: float64(1), object(13)
memory usage: 1.5+ MB


job_title          0
company            0
job_location       0
job_link           0
first_seen         0
search_city        0
search_country     0
job level          0
job_type           0
job_description    0
job_skills         0
job_skills_list    0
job_text           0
match_score        0
dtype: int64

In [14]:
postings_clean["first_seen"] = pd.to_datetime(
    postings_clean["first_seen"], errors="coerce"
)


In [15]:
postings_clean["job_skills_list"] = postings_clean["job_skills"].apply(
    lambda x: [s.strip().lower() for s in str(x).split(",") if s.strip()]
)

postings_clean[["job_title", "job_skills_list"]].head()


,job_title,job_skills_list
1,Market Research & Insights Analyst,"[data analysis, market research, survey develo..."
2,Business Systems Analyst `1,"[business analysis, technical writing, softwar..."
3,Senior VAT and Indirect Tax Analyst,"[accounting, finance, vat/gst tax regimes, us ..."
4,Senior HRIS Analyst (Timekeeping and Payroll),"[workday hcm, ukg dimensions, ceridian dayforc..."
5,Business Intelligence Reporting Analyst 2,"[sap business objects, sql, qlik, data modelin..."


In [16]:
postings_clean["job_text"] = (
    postings_clean["job_title"].fillna("") + " " +
    postings_clean["job_description"].fillna("") + " " +
    postings_clean["job_skills"].fillna("")
)

postings_clean["job_text"].head()


1    Market Research & Insights Analyst Company Des...
2    Business Systems Analyst `1 Overview The Busin...
3    Senior VAT and Indirect Tax Analyst We're look...
4    Senior HRIS Analyst (Timekeeping and Payroll) ...
5    Business Intelligence Reporting Analyst 2 Over...
Name: job_text, dtype: object

In [17]:
postings_clean.to_csv("Data Sets/cleaned/postings_clean.csv", index=False)
print("Saved updated postings_clean.csv")


Saved updated postings_clean.csv


In [18]:
postings_clean.rename(columns={"job level": "job_level"}, inplace=True)


In [19]:
postings_clean.to_csv("Data Sets/cleaned/postings_clean.csv", index=False)


In [20]:
def skill_match_score(resume_skills, job_skills):
    resume_set = set([s.lower() for s in resume_skills])
    job_set = set(job_skills)

    if len(job_set) == 0:
        return 0

    return len(resume_set.intersection(job_set)) / len(job_set)


In [21]:
resume_skills = ["python", "sql", "excel", "tableau"]

postings_clean["skill_score"] = postings_clean["job_skills_list"].apply(
    lambda x: skill_match_score(resume_skills, x)
)

postings_clean.sort_values("skill_score", ascending=False)[
    ["job_title", "skill_score"]
].head(10)


,job_title,skill_score
1640,Data Analyst,1.000000
1039,Senior Data Analyst,0.666667
5294,Data Analyst-III #: 23-06518,0.600000
736,Senior Financial Analyst,0.500000
8322,Healthcare Data Analyst,0.500000
2846,Financial Systems Analyst,0.500000
2277,Sr Data/Business Analyst,0.500000
9645,Treasury Analyst Senior,0.500000
72,Senior Data Analyst,0.500000
6799,Data Analyst I,0.500000


In [22]:
def matched_and_missing(resume_skills, job_skills):
    resume_set = set([s.lower().strip() for s in resume_skills])
    job_set = set(job_skills)

    matched = sorted(list(resume_set.intersection(job_set)))
    missing = sorted(list(job_set - resume_set))

    score = len(matched) / len(job_set) if len(job_set) > 0 else 0
    return score, matched, missing


In [23]:
resume_skills = ["python", "sql", "excel", "tableau"]

postings_clean[["skill_score", "matched_skills", "missing_skills"]] = postings_clean["job_skills_list"].apply(
    lambda js: pd.Series(matched_and_missing(resume_skills, js))
)

postings_clean.sort_values("skill_score", ascending=False)[
    ["job_title", "company", "skill_score", "matched_skills", "missing_skills"]
].head(5)


,job_title,company,skill_score,matched_skills,missing_skills
1640,Data Analyst,Adame Services,1.000000,"[sql, tableau]",[]
1039,Senior Data Analyst,Proven Recruiting,0.666667,"[sql, tableau]",[gcp bigquery]
5294,Data Analyst-III #: 23-06518,HireTalent - Diversity Staffing & Recruiting Firm,0.600000,"[excel, sql, tableau]","[google sheets, sigma]"
736,Senior Financial Analyst,Top Stack,0.500000,[excel],[fp&a experience]
8322,Healthcare Data Analyst,Public Consulting Group,0.500000,[sql],[sas]


In [24]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


In [25]:
tfidf = TfidfVectorizer(stop_words="english")

job_vectors = tfidf.fit_transform(postings_clean["job_text"])


In [26]:
resume_text = "Python SQL Excel Tableau data analysis dashboard reporting"
resume_vector = tfidf.transform([resume_text])


In [27]:
similarity_scores = cosine_similarity(resume_vector, job_vectors).flatten()


In [28]:
postings_clean["similarity_score"] = similarity_scores

postings_clean.sort_values("similarity_score", ascending=False)[
    ["job_title", "company", "similarity_score"]
].head(10)

,job_title,company,similarity_score
2418,Data Analyst //Pay rate: $43.35/hr,Stellar Professionals,0.426760
1598,Data Dashboard Analyst,Cypress HCM,0.418465
1639,Data Analyst,Adame Services,0.392926
1640,Data Analyst,Adame Services,0.380168
2211,Data/reporting analyst,"Global Channel Management, Inc.",0.325846
3364,Data Analyst (Candidate must currently reside ...,"Veridian Tech Solutions, Inc.",0.307037
6095,Sr./ Jr. Tableau Developer/Data Analyst _ Remote,Ekodus INC.,0.306342
5661,Supply Chain Data Analyst - Remote | WFH,Get It Recruit - Transportation,0.304900
1593,Data Analyst,Global Accounting Network,0.304449
9229,Data Analyst //Pay rate: $43.35/hr,Stellar Professionals,0.301914


In [29]:
import sys
print(sys.executable)


c:\Projects\AI -Job Recommendation System (NLP)\.venv\Scripts\python.exe


In [30]:
import sklearn
print(sklearn.__version__)


1.8.0


In [31]:
postings_clean[["job_title", "job_skills", "job_skills_list"]].head()


,job_title,job_skills,job_skills_list
1,Market Research & Insights Analyst,"Data analysis, Market research, Survey develop...","[data analysis, market research, survey develo..."
2,Business Systems Analyst `1,"Business Analysis, Technical Writing, Software...","[business analysis, technical writing, softwar..."
3,Senior VAT and Indirect Tax Analyst,"Accounting, Finance, VAT/GST tax regimes, US a...","[accounting, finance, vat/gst tax regimes, us ..."
4,Senior HRIS Analyst (Timekeeping and Payroll),"Workday HCM, UKG Dimensions, Ceridian Dayforce...","[workday hcm, ukg dimensions, ceridian dayforc..."
5,Business Intelligence Reporting Analyst 2,"SAP Business Objects, SQL, Qlik, Data Modeling...","[sap business objects, sql, qlik, data modelin..."


In [32]:
postings_clean.sort_values("skill_score", ascending=False)[
    ["job_title", "company", "skill_score", "matched_skills", "missing_skills"]
].head()


,job_title,company,skill_score,matched_skills,missing_skills
1640,Data Analyst,Adame Services,1.000000,"[sql, tableau]",[]
1039,Senior Data Analyst,Proven Recruiting,0.666667,"[sql, tableau]",[gcp bigquery]
5294,Data Analyst-III #: 23-06518,HireTalent - Diversity Staffing & Recruiting Firm,0.600000,"[excel, sql, tableau]","[google sheets, sigma]"
736,Senior Financial Analyst,Top Stack,0.500000,[excel],[fp&a experience]
8322,Healthcare Data Analyst,Public Consulting Group,0.500000,[sql],[sas]


In [33]:
it_keywords_strict = [
    "software", "developer", "programmer", "engineer",
    "cloud", "aws", "azure", "gcp",
    "devops", "sre",
    "cyber", "security", "soc", "siem",
    "network", "routing", "switching", "firewall",
    "systems", "system administrator", "sysadmin",
    "linux", "windows server", "active directory",
    "database administrator", "dba",
    "full stack", "backend", "frontend",
    "java", "c++", "c#", ".net", "javascript", "react", "node",
    "kubernetes", "docker", "terraform"
]

In [34]:
business_exclude = [
    "business analyst", "market", "sales", "accounting", "finance",
    "tax", "audit", "hr", "payroll", "procurement", "supply chain",
    "customer service", "marketing", "operations"
]

In [35]:
import re

title = postings_clean["job_title"].str.lower().fillna("")
desc  = postings_clean["job_description"].str.lower().fillna("")

it_match = title.str.contains("|".join(map(re.escape, it_keywords_strict))) | \
           desc.str.contains("|".join(map(re.escape, it_keywords_strict)))

business_match = title.str.contains("|".join(map(re.escape, business_exclude)))

postings_it = postings_clean[it_match & (~business_match)].copy()

print("Total IT-focused jobs:", len(postings_it))
postings_it["job_title"].head(25)

Total IT-focused jobs: 7970


2                           Business Systems Analyst `1
5             Business Intelligence Reporting Analyst 2
10                                          GIS Analyst
11                          Sr Business Systems Analyst
12                         Senior Network Analyst - DDI
13                          Sr Business Systems Analyst
15              Senior Network Analyst - Campus Network
19                   Systems Analyst I, II, III, Senior
22             Senior Business & Legal Research Analyst
24    Management Analyst III Agricultural Survey Sta...
25                            Fiscal Management Analyst
26                            Fiscal Management Analyst
28                        Senior SAP PP and P2P Analyst
30    Systems Administration Senior Analyst This pos...
31    Volunteer: Wildfire Prevention Financing Data ...
32                   Systems Analyst Programmer, Senior
33                   Systems Analyst Programmer, Senior
34                             Senior Financial 

In [36]:
postings_it["job_title"].value_counts().head(20)

job_title
Data Analyst                                                           253
Senior Data Analyst                                                    202
Business Systems Analyst                                               199
Senior Financial Analyst                                               176
Business Intelligence Analyst                                          115
35F Intelligence Analyst                                                82
Business System Analyst                                                 78
Lead Analyst Identity Governance and Administration (SailPoint IIQ)     73
Data Catalog and Lineage Central Governance Analyst                     61
Business Data Analyst                                                   58
Sr. Data Analyst                                                        47
Analyst II, Credit                                                      36
Data Research Analyst, gt.school (Remote) - $60,000/year USD            35
Lead Data Analy

In [37]:
it_keywords = [
    "data analyst",
    "data engineer",
    "data scientist",
    "machine learning",
    "ai ",
    "python",
    "sql",
    "tableau",
    "power bi",
    "analytics",
    "database",
    "etl",
    "big data",
    "cloud",
    "aws",
    "azure",
    "gcp"
]

exclude_keywords = [
    "finance",
    "financial",
    "accounting",
    "tax",
    "audit",
    "payroll",
    "marketing",
    "sales",
    "procurement",
    "supply chain",
    "hr",
    "human resources"
]

In [38]:
title = postings_clean["job_title"].str.lower().fillna("")
desc = postings_clean["job_description"].str.lower().fillna("")

it_match = title.str.contains("|".join(it_keywords)) | \
           desc.str.contains("|".join(it_keywords))

exclude_match = title.str.contains("|".join(exclude_keywords))

postings_it = postings_clean[it_match & (~exclude_match)].copy()

print("Total IT jobs:", len(postings_it))

Total IT jobs: 7912


In [39]:
postings_it["job_title"].value_counts().head(20)

job_title
Data Analyst                                                           323
Senior Data Analyst                                                    230
Business Analyst                                                       209
Business Intelligence Analyst                                          134
Business Systems Analyst                                               119
Business Data Analyst                                                   73
Lead Analyst Identity Governance and Administration (SailPoint IIQ)     73
Senior Associate, Senior Security Operations Analyst                    71
Data Catalog and Lineage Central Governance Analyst                     61
Business System Analyst                                                 52
Sr. Data Analyst                                                        51
US Tech - Business Analyst                                              41
Technical Business Analyst                                              40
US Tech Sr. Bus

In [40]:
postings_it.to_csv("Data Sets/cleaned/postings_it.csv", index=False)

In [41]:
# SAMPLE RESUME TEXT

resume_text = """
Data Analyst with skills in Python, SQL, Excel and Tableau.
Experience in data cleaning, visualization and reporting.
Knowledge of machine learning basics.
"""

resume_text

'\nData Analyst with skills in Python, SQL, Excel and Tableau.\nExperience in data cleaning, visualization and reporting.\nKnowledge of machine learning basics.\n'

In [42]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [43]:
# TF-IDF MODEL

vectorizer = TfidfVectorizer(stop_words='english')

tfidf_matrix = vectorizer.fit_transform(
    postings_it["job_text"]
)

resume_vector = vectorizer.transform([resume_text])

similarity_scores = cosine_similarity(
    resume_vector,
    tfidf_matrix
).flatten()

similarity_scores[:5]

array([0.04552013, 0.00962729, 0.07227773, 0.01928847, 0.02531393])

In [44]:
postings_it["similarity_score"] = similarity_scores

postings_it.sort_values(
    "similarity_score",
    ascending=False
)[
["job_title","company","similarity_score"]
].head(10)

,job_title,company,similarity_score
1604,Data Analyst,Stellar Professionals,0.312597
1541,Data Analyst,Infotree Global Solutions,0.291553
8454,Part-Time Data Analyst (Remote),Progilisyssolutions,0.271858
1593,Data Analyst,Global Accounting Network,0.270737
1639,Data Analyst,Adame Services,0.259656
8056,Sr. Data Analyst,Florida Health Care Plans,0.257253
6426,Customer Service Representative/Data Analyst/D...,Louisvuitton,0.247216
221,Sr. Data Analyst,Creative Circle,0.243724
1644,Data Analyst,Curate Insights,0.238888
4150,Senior Data Analyst - Remote | WFH,Get It Recruit - Information Technology,0.238270
